# Lab 9.5 &mdash; Challenge: The Service That Is Up and Wrong

**Level:** Advanced &middot; challenge &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 3 &middot; Module 9 &mdash; Deployment &amp; AgentOps**

### What you'll do
- Find an incident in which every conventional signal is green
- Decide which direction each metric has to move before it is worth a page
- Pick a scaling signal that moves when CPU does not
- Leave with the runbook page for an agentic service

> **How this lab works.** You write real FastAPI, Pydantic, LangChain and Kubernetes-manifest
> code. Fill every `BLANK`, then run the **Self-check** cell under each section &mdash; those
> assert on the *objects you built* (a route table, a request contract, a compiled tool, a
> manifest dict), so they are deterministic. **No graded cell needs a cluster, a running server
> or a model.** Cells marked **Run it for real** put your code in front of the sandbox model,
> your own namespace or the tracing backend; if any of those is unreachable they print how to
> fix it instead of crashing. The score line is feedback, not a grade.

> **The last lab of the course.** It uses Module 6's citations, Module 7's measurements
> and Module 8's controls, and asks the Module 9 question about all three: how would
> you know, at 09:15, from a dashboard?

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap
from typing import Any, Callable, Optional

WORK = os.path.join("/tmp", "awmas-lab-9-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model can reason before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because a deployment lab makes a lot of small calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL, api_key=LLM_API_KEY,
                          temperature=temperature, extra_body=NO_THINK)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- your own namespace --------------------------------------------------
# You deploy into your own namespace, published at your own host. Both are injected into
# the sandbox, so nothing here is hardcoded and nothing here needs them to be set.
#
# Read ONLY from APP_NAMESPACE, never derived from the hostname. A cell below runs
# kubectl against whatever this says, and a namespace guessed from a machine name is
# the wrong thing to point kubectl at.
APP_NS   = os.environ.get("APP_NAMESPACE", "")
APP_HOST = os.environ.get("APP_HOST", "")

print("work dir :", WORK)
print("model    :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")
print("namespace:", APP_NS or "(unknown -- no graded cell needs it)")

## Concept

An ordinary service fails by erroring or by slowing down, and both are visible in the four golden
signals &mdash; latency, traffic, errors, saturation. An agent service has a third failure mode:
it answers every request, quickly, with a 200, and the answers are wrong.

Nothing in the golden signals moves. Something else does, and only if you recorded it.

## Section 1 &mdash; Two days that look identical

Here is yesterday and today. Same traffic, same code, one deploy in between. `RequestRecord`
is the telemetry schema from Lab 9.4 &mdash; note which two fields exist only because somebody
chose to record them.

In [ ]:
import random
from pydantic import BaseModel

class RequestRecord(BaseModel):
    """One request, as your telemetry recorded it.

    Which fields exist here IS the instrumentation decision from Lab 9.4. The first four
    any HTTP service records for free; the last two exist only because somebody chose to
    record what the agent DECIDED and whether the answer was grounded.
    """
    ok: bool
    duration_s: float
    cost_usd: float
    decision: str            # answered | refused | escalated
    cited: bool


def build_day(seed: int, refusal_rate: float, citation_rate: float,
              error_rate: float = 0.02, n: int = 2000) -> list:
    """One day of requests. The same seed gives the same latencies, costs and errors,
    so any difference between two days below is a difference in BEHAVIOUR, not noise."""
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        r = rng.random()
        decision = ("refused" if r < refusal_rate
                    else "escalated" if r < refusal_rate + 0.04
                    else "answered")
        out.append(RequestRecord(
            ok=rng.random() > error_rate,
            duration_s=round(rng.uniform(1.5, 12.0), 2),
            cited=rng.random() < citation_rate,
            cost_usd=round(rng.uniform(0.0008, 0.0032), 5),
            decision=decision))
    return out


YESTERDAY = build_day(11, refusal_rate=0.08, citation_rate=0.92)
TODAY     = build_day(11, refusal_rate=0.01, citation_rate=0.55)


def pct(values, p):
    """The p-th percentile, nearest-rank. Stdlib, and exact enough for a dashboard."""
    s = sorted(values)
    return s[min(len(s) - 1, max(0, math.ceil(p / 100 * len(s)) - 1))]


def metrics(day: list) -> dict:
    n = len(day)
    return {
        "requests":        n,
        "error_rate":      round(sum(1 for r in day if not r.ok) / n, 4),
        "p95_latency_s":   pct([r.duration_s for r in day], 95),
        "cost_per_req":    round(sum(r.cost_usd for r in day) / n, 5),
        "refusal_rate":    round(sum(1 for r in day if r.decision == "refused") / n, 4),
        "escalation_rate": round(sum(1 for r in day if r.decision == "escalated") / n, 4),
        "citation_rate":   round(sum(1 for r in day if r.cited) / n, 4),
    }


GOLDEN  = ("error_rate", "p95_latency_s", "cost_per_req", "requests")
AGENTIC = ("refusal_rate", "escalation_rate", "citation_rate")

print("recorded per request:", list(RequestRecord.model_fields))
print("yesterday:", metrics(YESTERDAY))
print("today    :", metrics(TODAY))

In [ ]:
def alarm_direction(metric: str) -> str:
    """Which way must this metric move before you want to be woken up?

    "up"   -- only an increase is bad
    "down" -- only a decrease is bad
    "both" -- either direction is a change in behaviour worth looking at
    """
    if metric in ("error_rate", "p95_latency_s", "cost_per_req"):
        return "up"                 # nobody is paged because errors fell
    if metric == "requests":
        return "both"               # traffic vanishing is an incident too
    # TODO: refusal_rate, escalation_rate and citation_rate are the OUTPUT OF A CONTROL --
    # a guardrail firing, a case going to a human, an answer being grounded. A control
    # that stops working makes its own metric FALL. Which direction do you need here?
    return BLANK


def moved(metric: str, before: float, after: float, tolerance: float = 0.25) -> bool:
    """Did this metric change materially, in a direction that matters for it?"""
    if before == 0:
        return after != 0
    change = (after - before) / before
    if alarm_direction(metric) == "up":
        return change > tolerance
    if alarm_direction(metric) == "down":
        return change < -tolerance
    return abs(change) > tolerance


def what_changed(before: dict, after: dict, tolerance: float = 0.25) -> list:
    """Every metric that moved, in the order they are defined."""
    return [k for k in before if moved(k, before[k], after[k], tolerance)]

In [ ]:
# --- Self-check: Section 1   (two dicts of numbers -- no cluster, no model)
Y, T = metrics(YESTERDAY), metrics(TODAY)

check("traffic is identical",
      lambda: Y["requests"] == T["requests"])
check("the error rate did not move",
      lambda: not moved("error_rate", Y["error_rate"], T["error_rate"]))
check("p95 latency did not move",
      lambda: not moved("p95_latency_s", Y["p95_latency_s"], T["p95_latency_s"]))
check("cost per request did not move",
      lambda: not moved("cost_per_req", Y["cost_per_req"], T["cost_per_req"]))
check("NOT ONE OF THE FOUR GOLDEN SIGNALS MOVED",
      lambda: not any(moved(k, Y[k], T[k]) for k in GOLDEN),
      "every dashboard the team owns is green")
check("the refusal rate collapsed",
      lambda: moved("refusal_rate", Y["refusal_rate"], T["refusal_rate"]))
check("...DOWNWARDS, which is why a one-sided alarm never fired",
      lambda: T["refusal_rate"] < Y["refusal_rate"])
check("the citation rate fell too",
      lambda: moved("citation_rate", Y["citation_rate"], T["citation_rate"]))
check("a control's metric is watched in both directions",
      lambda: alarm_direction("refusal_rate") == "both",
      "an increase means the guardrail got noisier; a fall means it stopped working")
check("...while latency is only watched upwards",
      lambda: alarm_direction("p95_latency_s") == "up",
      "nobody is paged because the service got faster")
check("exactly the agent-specific metrics moved, and only those",
      lambda: set(what_changed(Y, T)) == {"refusal_rate", "citation_rate"})
check("a one-sided test over everything finds nothing at all",
      lambda: [k for k in Y if T[k] > Y[k] * 1.25] == [],
      "which is how this runs for three weeks")

def _diff():
    print(f"  {'metric':18} {'dir':>5} {'yesterday':>10} {'today':>10}   moved?")
    for k in Y:
        flag = "  <-- MOVED" if moved(k, Y[k], T[k]) else ""
        print(f"  {k:18} {alarm_direction(k):>5} {Y[k]:>10} {T[k]:>10}{flag}")
guard(_diff)

### What happened

A deploy changed a prompt. The guardrail that used to hold sanctions cases for a human now
answers most of them, and the retriever's grounding check stopped rejecting ungrounded answers.

The service is up. It is fast. It costs the same. It answers every request with a 200, and one
payment in twelve that should have gone to a human did not.

This is the failure mode Module 8 closed on, seen from the dashboard, and the reason those two
metrics have to exist as **first-class signals with alarms on them**, next to latency and errors
rather than in a weekly report. Note also that neither of them exists unless `RequestRecord`
carries the field &mdash; Lab 9.4's decision, arriving three weeks later.

## Section 2 &mdash; The signal that actually moves with load

The second half of the incident: at 09:15 the same service went from four concurrent requests to
sixty, and the HPA did nothing at all.

In [ ]:
CALL_SECONDS   = 8.0      # one agent request, mostly spent waiting on the gateway.
                          # Measured on this sandbox: 7.5-10s for a one-line answer.
CPU_PER_REQ    = 0.015    # the CPU it actually uses
PER_REPLICA    = 8        # concurrent requests one replica serves without queueing
TARGET_UTIL    = 0.70
SCALING_SIGNALS = ("cpu", "memory", "requests_per_second", "in_flight")

def cpu_percent(in_flight: int) -> float:
    """CPU utilisation of the fleet's replicas at this concurrency."""
    return 100.0 * in_flight * CPU_PER_REQ / CALL_SECONDS


def replicas_from_cpu(in_flight: int, current: int = 1, target: int = 70) -> int:
    """What an HPA on CPU utilisation asks for. This is the shipped default."""
    util = cpu_percent(in_flight) / current
    return max(1, math.ceil(current * util / target))


def replicas_from_inflight(in_flight: int) -> int:
    """What an HPA on in-flight requests asks for: enough replicas to hold them all at
    TARGET_UTIL of what one replica serves without queueing."""
    return max(1, math.ceil(in_flight / (PER_REPLICA * TARGET_UTIL)))


def latency_at(in_flight: int, replicas: int) -> float:
    """Wall clock per request once the queue forms. Crude, and the right shape."""
    capacity = replicas * PER_REPLICA
    return CALL_SECONDS * math.ceil(max(1, in_flight) / capacity)


def scaling_signal() -> str:
    """Which of SCALING_SIGNALS should this service's HPA scale on?

    Run the table at the foot of this section before you answer.
    """
    # TODO: pick the one that GROWS while this service is overloaded. CPU sits at 16% while
    # requests queue for a minute. Memory is flat -- the process holds a few dictionaries.
    # Requests per second measures the ARRIVAL rate, which stays level while the queue
    # behind it grows, so it cannot tell a healthy minute from a saturated one.
    return BLANK


def replicas_for(signal: str, in_flight: int, current: int = 1) -> int:
    """The replica count each candidate signal would ask for."""
    if signal == "cpu":
        return replicas_from_cpu(in_flight, current)
    if signal == "in_flight":
        return replicas_from_inflight(in_flight)
    return current            # memory and arrival rate do not move: the fleet stays put

In [ ]:
# --- Self-check: Section 2
check("at four in flight one replica is right, and both rules agree",
      lambda: replicas_from_cpu(4) == 1 and replicas_from_inflight(4) == 1)
check("at sixty in flight the CPU rule still asks for one",
      lambda: replicas_from_cpu(60) == 1)
check("...because the fleet is under 16% busy while it queues",
      lambda: cpu_percent(60) < 16)
check("THE IN-FLIGHT RULE ASKS FOR ELEVEN",
      lambda: replicas_from_inflight(60) == 11)
check("one replica at sixty in flight is a 64-second request",
      lambda: latency_at(60, 1) == 64.0)
check("eleven replicas bring it back to one call time",
      lambda: latency_at(60, replicas_from_inflight(60)) == CALL_SECONDS)
check("you picked a signal that grows when this service is overloaded",
      lambda: scaling_signal() == "in_flight",
      "cpu and memory are flat; requests_per_second is the ARRIVAL rate, which stays "
      "level while the queue behind it grows")
check("...and it is one of the four candidates",
      lambda: scaling_signal() in SCALING_SIGNALS)
check("your signal scales out at sixty in flight; CPU does not",
      lambda: replicas_for(scaling_signal(), 60) > replicas_for("cpu", 60))
check("the CPU rule leaves latency 8x worse than yours",
      lambda: latency_at(60, replicas_for("cpu", 60))
              == 8 * latency_at(60, replicas_for(scaling_signal(), 60)))
check("no rule scales down below one replica",
      lambda: replicas_from_cpu(0) == 1 and replicas_from_inflight(0) == 1)
check("scaling is bounded by maxReplicas, which is a budget decision",
      lambda: min(replicas_from_inflight(400), 3) == 3,
      "at 400 in flight it wants 72; your quota says 3, so the answer is a queue and a 429")

def _scaling():
    print(f"  {'in flight':>10} {'CPU %':>7} {'cpu rule':>9} {'inflight rule':>14} "
          f"{'latency (cpu)':>14} {'latency (inflight)':>19}")
    for n in (4, 12, 30, 60, 120):
        rc, ri = replicas_from_cpu(n), replicas_from_inflight(n)
        print(f"  {n:>10} {cpu_percent(n):>6.1f}% {rc:>9} {ri:>14} "
              f"{latency_at(n, rc):>13.0f}s {latency_at(n, ri):>18.0f}s")
guard(_scaling)

### Read it

The HPA in the starter manifest &mdash; and in most agent deployments &mdash; scales on CPU at
70%. On this workload it reaches 16% at sixty concurrent requests, so it never fires, and the
replica sitting at 16% busy is serving 64-second requests.

The fix is not a lower CPU target. It is a **different signal**: in-flight requests, queue depth,
or time-to-first-token, exported by your own service and scraped as a custom metric. All three
grow with load because all three are about waiting, which is what this service does.

And note the last check. Scaling has a ceiling that is a budget, not a technical limit. Past it,
the correct behaviour is to shed load with a `429` and a `Retry-After`, not to accept a request
you will answer in four minutes.

## Section 3 &mdash; Alarms that would have caught it

An alarm has two jobs, and the second one is why most alarms get switched off: fire on the
incident, and stay quiet on every good day.

In [ ]:
def alarm_golden_signals(before: dict, after: dict) -> bool:
    """The alarms the team already has. Provided so you can see them not fire."""
    return any(moved(k, before[k], after[k]) for k in GOLDEN)


def alarm_control_drift(before: dict, after: dict) -> bool:
    """A control's own metric moved, in whichever direction alarm_direction allows."""
    return any(moved(k, before[k], after[k]) for k in AGENTIC)


def alarm_saturation(in_flight: int, replicas: int) -> bool:
    """Fires while requests are queueing, whatever the CPU says."""
    capacity = replicas * PER_REPLICA
    # TODO: you do not want to be paged at 100% of capacity -- by then every new request
    # is already waiting. Which fraction of capacity should this fire at? Use the SAME
    # number the scaling rule uses, so the alarm and the autoscaler cannot disagree.
    return in_flight > capacity * BLANK

In [ ]:
# --- Self-check: Section 3
QUIET_DAY = metrics(build_day(12, refusal_rate=0.08, citation_rate=0.92))

check("the alarms the team already has do not fire on the incident",
      lambda: alarm_golden_signals(Y, T) is False,
      "this is not a criticism of them -- they are measuring something else")
check("THE CONTROL-DRIFT ALARM FIRES",
      lambda: alarm_control_drift(Y, T) is True)
check("...and stays quiet comparing two ordinary days",
      lambda: alarm_control_drift(Y, QUIET_DAY) is False,
      "an alarm that fires on a good day is an alarm somebody mutes")
check("the golden-signal alarms are also quiet on a good day",
      lambda: alarm_golden_signals(Y, QUIET_DAY) is False)
check("the saturation alarm fires at sixty in flight on one replica",
      lambda: alarm_saturation(60, 1) is True)
check("...and not once it has scaled out",
      lambda: alarm_saturation(60, replicas_from_inflight(60)) is False)
check("it fires before latency doubles, not after",
      lambda: alarm_saturation(9, 1) is True and latency_at(9, 1) == 2 * CALL_SECONDS)
check("it agrees with the autoscaler about what full means",
      lambda: all(alarm_saturation(n, 1) == (replicas_from_inflight(n) > 1)
                  for n in (4, 6, 8, 12, 30)),
      "an alarm at a different threshold from the scaler pages you about work already in hand")
check("a CPU alarm at 70% is silent at every concurrency worth alarming on",
      lambda: all(cpu_percent(n) < 70 for n in (10, 60, 120, 200)),
      "it crosses 70% only near 400 in flight, where a request already takes 400 seconds")

def _alarms():
    for label, pair in (("incident (yesterday -> today)", (Y, T)),
                        ("ordinary day vs ordinary day",  (Y, QUIET_DAY))):
        print(f"  {label:32} golden={str(alarm_golden_signals(*pair)):5} "
              f"control_drift={alarm_control_drift(*pair)}")
    print()
    for n, reps in ((4, 1), (60, 1), (60, 11)):
        print(f"  {n:>3} in flight on {reps:>2} replica(s): saturation="
              f"{str(alarm_saturation(n, reps)):5} latency={latency_at(n, reps):.0f}s")
guard(_alarms)

## The runbook page

Everything above is one page of an on-call runbook. Yours will differ; the shape will not.

**Page on**

| Signal | Threshold | Because |
|---|---|---|
| 5xx rate | above 1% for 5 min | the ordinary one; keep it |
| p95 latency | above 3&times; the baseline | the ordinary one; keep it |
| in-flight per replica | above 70% of capacity | CPU will not tell you this |
| refusal / escalation rate | moved &plusmn;25% vs the last 7 days | **a control stopped working** |
| citation rate | moved &plusmn;25% | grounding stopped working |
| cost per request | above 2&times; the baseline | a retry loop, or a routing change |

**First three things to do**

1. **Read one trace, not the logs.** Find a slow or wrong request by trace ID and look at the
   span tree. Which hop grew, and did it grow in count or in duration?
2. **Compare the last deploy.** A prompt is a deploy. So is a model version change made by
   somebody else on the gateway you depend on.
3. **Check the dependency before restarting anything.** Lab 9.2's whole point: restarting a
   healthy process because a remote gateway blinked makes the outage longer.

**Do not**

- Do not raise the CPU target to make the HPA fire. It is the wrong signal, not a mistuned one.
- Do not turn off the control that is alarming. Its metric moving is the alarm.
- Do not conclude anything from a green dashboard. Today's incident had one.

## Run it for real

Your own numbers, from the sandbox gateway. Four sequential calls, then the replica count your
chosen signal would ask for at sixty concurrent users.

In [ ]:
if llm_ready():
    def _budget():
        lat = []
        for i in range(4):
            t0 = time.perf_counter()
            reply = ask(f"In one sentence, what is a payment exception? (v{i})")
            if reply.startswith("<model unavailable"):
                print(reply)                  # say so, rather than timing a failure
                return
            lat.append(time.perf_counter() - t0)
        p95 = pct(lat, 95)
        want = replicas_for(scaling_signal(), 60)
        print(f"  measured  : mean {sum(lat) / len(lat):.1f}s, p95 {p95:.1f}s over 4 calls")
        print(f"  at 60 concurrent users, signal={scaling_signal()!r} asks for {want} replicas")
        print(f"  the CPU rule asks for {replicas_for('cpu', 60)}")
        print(f"  and one replica would answer in about {latency_at(60, 1):.0f}s")
        print("\n  Four calls is not a latency distribution. It is enough to know which")
        print("  order of magnitude you are budgeting in, which is the decision here.")
    guard(_budget)

In [ ]:
score()

## Your turn

1. The control-drift alarm compares two windows. Write the version that compares today with a
   **trailing seven-day median**, and work out what it does on the Monday after a long weekend.
2. Add a `429` path to Lab 9.1's `ask_endpoint`: shed load when in-flight is above capacity,
   with a `Retry-After`. Then decide which is worse for your callers &mdash; a 429 now, or a 200
   in four minutes.
3. Take one control from Module 8 &mdash; the approval gate, the contract, the detector &mdash;
   and add the field to `RequestRecord` that proves it is still running. If you cannot name one,
   that control is unmonitored, and Section 1 is what that looks like on the day it stops.

**What you take from Module 9:** a service boundary with typed contracts and an approval gate,
probes that answer two different questions, a readiness checklist that executes, spans that carry
cost and decisions, and the two signals &mdash; control drift and saturation &mdash; that an
agent needs and a web service does not.

That is the last lab. The capstone puts all nine modules behind one endpoint.